# 05 · 代理（Agents）

当任务步骤不可预知（需要「看情况」决定下一步）时，用 **Agent** 让 LLM 自己决定调用哪些工具、以什么顺序。本章定义工具并让模型自主选择调用。

运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已配置 `.env` 中的 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/05-agents.md`。


## 0. 环境检查


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 定义工具

用 `@tool` 装饰器把普通函数变成工具，函数的 **docstring 会被模型用来决定何时调用**。这里定义一个计算器（限制命名空间避免执行任意代码）和一个获取当前时间的工具。


In [ ]:
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))

@tool
def calculator(expression: str) -> str:
    """计算一个四则运算表达式，例如 '23 * 7 + 4'。"""
    try:
        # 限制命名空间，避免执行任意代码
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'计算错误：{e}'

@tool
def get_current_time() -> str:
    """返回当前本地时间，当用户询问时间时调用。"""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

tools = [calculator, get_current_time]


## 2. 构造 Agent（必须包含 agent_scratchpad 占位符）

`create_tool_calling_agent` 基于模型的「工具调用（function calling）」能力，比旧版 ReAct 文本解析更稳定。`MessagesPlaceholder("agent_scratchpad")` 用于存放中间思考过程。


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个会使用工具的助手，必要时调用工具来回答。'),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)


## 3. 运行：模型自行决定调用哪个工具、以什么顺序


In [ ]:
answer = agent_executor.invoke(
    {'input': '现在是几点？另外帮我算一下 23 乘以 7 是多少。'}
)['output']
print('\n最终回答：', answer)


## 4. 下一步

- 服务化部署：运行 `examples/06_langserve.py`，参见 `docs/06-langserve-and-deployment.md`

**常见坑**：工具描述写得太含糊，模型不会调用或调错；务必设置 `max_iterations` 避免无限循环；工具内部异常要兜底，否则中断整个 Agent。
